In [1]:
Dnefrom copy import deepcopy
from datetime import date, timedelta
import json
from time import sleep

import requests

from MMutils.handlers.secrets_handler import SecretsManager

In [2]:
secret_handler = SecretsManager(
    project_id="rs-marketmaking"
)
opticodds_access_key = secret_handler.get_secret(
    "opticodds_access_key"
    # "opticodds_parlay_tennis_ucl_access_key"
)

headers = {
    'accept': 'application/json',
    'X-Api-Key': opticodds_access_key
}

In [3]:
ENCODING = "utf8"

def get_optic(url, params=None):
    params = deepcopy(params) if params else {}

    datas = []
    page = 1
    while True:
        params["page"] = str(page)
        response = requests.get(url, params, headers=headers)
        data = response.json()
        datas += data["data"]
        if data.get("total_pages", 1) <= page:
            break
        page += 1

    return datas

In [4]:
# sports = get_optic(
#     "https://api.opticodds.com/api/v3/"
#     "sports/active")

# for sport in sports:
#     print(sport["id"])

In [5]:
# leagues = get_optic(
#     "https://api.opticodds.com/api/v3/"
#     "leagues/active?sport=hockey")

# for league in leagues:
#     print(league["id"], league["name"], league["region"], league["gender"])

In [14]:
LEAGUES = [
    # Basketball:
    # "nba",
    "ncaab",

    # Hockey:
    # "nhl",

    # Soccer:
    # "england_-_premier_league",
    # "france_-_ligue_1",
    # "germany_-_bundesliga",
    # "italy_-_serie_a",
    # "spain_-_la_liga",
    # "usa_-_major_league_soccer",
    # "uefa_-_champions_league",

    # MMA:
    # "ufc",
]

In [15]:
# sportsbooks = [
#     sb
#     for league in LEAGUES
#     for sb in get_optic(
#         "https://api.opticodds.com/api/v3/"
#         f"sportsbooks/active?league={league}")
# ]

# sorted(sb["id"] for sb in sportsbooks)

In [16]:
N_DAY = 1

start_date_before = (date.today() + timedelta(days=N_DAY)).isoformat()

fixtures = [
    fixture
    for league in LEAGUES
    for fixture in get_optic(
        "https://api.opticodds.com/api/v3/"
        f"fixtures/active?league={league}"
        f"&start_date_before={start_date_before}")
]

len(fixtures)

48

In [17]:
def clean_odds(o1, o2):
    assert o1.keys() == o2.keys()
    output = {}
    for k, v1 in o1.items():
        v2 = o2[k]
        if v1 == v2:
            v = v1
        elif len(v1) > len(v2):
            v = v1
        elif len(v2) > len(v1):
            v = v2
        else:
            assert False, (k, v1, v2)

        output[k] = v

    return output

In [18]:
fids = [f["id"] for f in fixtures]
fixture_odds = []

len(fids)

48

In [19]:
while fids:
    fid = fids[0]
    print(len(fids), fid, end=": ")
    fodds, = get_optic(
        "https://api.opticodds.com/api/v3/fixtures/odds?"
        f"fixture_id={fid}"
        "&sportsbook=kalshi"
        "&sportsbook=prophet_x"
        "&sportsbook=fanatics"
        "&sportsbook=draftkings"
        "&sportsbook=betmgm"
    )

    odds = deepcopy(fodds["odds"])
    odds_map = {}
    for o1 in odds:
        gid, sbid, mid, sid = o1["id"].split(":")
        clean_id = ":".join((gid, mid, sid))
        o1["id"] = clean_id

        del o1["is_main"]
        del o1["price"]
        del o1["timestamp"]
        o1.pop("deep_link", None)
        del o1["limits"]
        del o1["order_book"]

        sb = o1.pop("sportsbook")
        source_ids = o1.pop("source_ids")

        o2 = odds_map.get(clean_id, o1)
        clean_source_ids = o2.pop("source_ids", {})
        o3 = clean_odds(o1, o2)
        o3["source_ids"] = clean_source_ids
        odds_map[clean_id] = o3

        assert sb not in clean_source_ids
        if sb in ("Kalshi", "Prophet X") and source_ids:
            print(sb[0], end="")
            clean_source_ids[sb] = source_ids

    print()

    fodds["odds"] = list(odds_map.values())
    fixture_odds.append(fodds)

    sleep(0.2)
    del fids[0]

len(fixture_odds)

48 20260303392E8E34: PPPPKKKPPPKKPKPKKPPPP
47 202603033ABDDBC7: PPKPPKPPPPKPPPPPKPKPKKKPPP
46 20260303ACD17158: KPKPKPKKPPKKPPKKKKKPKKKPPKPKKKKPKPKPKKPPPKPPKPPPPPPPP
45 20260303D9997750: PKKPKKPKPPPKPPPKPPKPKKPP
44 202603030CD29E08: PPPPPKKKPPKPPKPKPPPKKPKPKPPP
43 202603036D92F5EC: PKPKKKPKKPPKPPKKKKPKKPKPPPKPKKKKKKPKKKKPKKPPKKKKPKPPKPPPKP
42 20260303452DD7FA: PPKPPPKKKPPPKPPPKPKPKPPKPPPPKPPKKPPKKPPKPKKK
41 2026030402926B6C: PPPKPKKPKPPPKKPPPP
40 2026030413344CAC: PKPKKKKKPPPKPKPKKKPPKKPKPKPKPKPPKKKPKK
39 20260304197229E8: PPPPPPPPPP
38 202603042354D605: PPKKKPPKPPPKPPKPKKKPPKPKKPKPPKKKPKKKPKKPKKKPKKKK
37 202603043BB5FADF: KKKPKKPPKPKKPPKPPPKKPPPKPKPKKKPKPKPKKP
36 202603043F99C296: KKKPPPPKPPKKKKKKPKKPKPKKKKPPPPPPKKKKKKKP
35 20260304661EEBF3: KPPPPPKPKKPKPPKPKKKKPPPKKPKPPKKPKPPKKP
34 2026030466ACB799: PPKKPPKKKPPKKKPKPKPPPPKKPPKPK
33 202603046DAFD708: KPKKPKKKPKPPPPPPKPPPKKKPKKPKPPPP
32 202603046EFBA99C: PPKPPPPPPPPKPKKP
31 2026030472878354: PPPPKPPKKKKPPPPPKKKKPK
30 2026030476D69520: 

48

In [20]:
with open(
        "/dump/main/wsd/mm/data/greg/optic_fixtures_odds.jsonl", "w",
        encoding=ENCODING) as file:
    for f in fixture_odds:
        print(json.dumps(f), file=file)